# threadlepy tutorial

This notebook mirrors the public workflow of `YukunJiao/threadleR`, but uses Python wrappers from `threadlepy`. It loads bundled example data, explores nodes and layers, edits a scratch network, and demonstrates import/export.

You need a working Threadle CLI executable. This tutorial assumes `threadle` is already on `PATH`.

In [ ]:
import os

import threadlepy
import threadlepy.commands as th

threadlepy.__version__, threadlepy.is_available()

## Start Threadle

Start a background Threadle process in JSON mode. The client sends one JSON command per wrapper call and returns the backend payload.

For scripts, `with threadlepy.session():` is the most convenient lifecycle helper. In a notebook, explicit `start()` / `stop()` cells make it easier to rerun sections interactively.

In [ ]:
threadlepy.configure(print_messages=False, timeout=1800)
threadlepy.start()
tutorial_workdir = th.get_workdir()
tutorial_workdir

## Load example data

`load_examples()` loads the package data under `src/threadlepy/Examples`. The returned objects are lightweight handles that store the backend variable name.

In [ ]:
examples = th.load_examples("mynet")
mynet = examples["mynet"]
mynet_nodeset = examples["mynet_nodeset"]

# `load_examples()` switches Threadle to the example-data folder.
# Switch back so Threadle file exports and Python open() use the same folder.
th.set_workdir(tutorial_workdir)

th.i(), th.info(mynet), th.info(mynet_nodeset)

In [ ]:
th.get_workdir()
th.dir()

## Nodes and attributes

Node commands work on either a nodeset handle or a network handle, since Threadle networks carry an underlying nodeset.

In [ ]:
th.get_nbr_nodes(mynet)
th.get_all_nodes(mynet, offset=0, limit=8)
th.get_nodeid_by_index(mynet, index=0)
th.get_random_node(mynet)

In [ ]:
th.get_attr(mynet_nodeset, nodeid=123, attrname="gender")
th.get_attrs(mynet_nodeset, nodes=[123, 234, 345], attrname="gender")
th.get_attr_summary(mynet_nodeset, attrname="gender")

## One-mode layers

`mynet` includes one-mode layers named `kinship` and `trade`. Use them for edge, alter, degree, density, component, and shortest-path commands.

In [ ]:
th.get_edge(mynet, "trade", node1id=123, node2id=345)
th.check_edge(mynet, "kinship", node1id=123, node2id=345)
th.get_all_edges(mynet, "trade", offset=0, limit=10)
th.get_random_edge(mynet, "trade")

In [ ]:
th.get_node_alters(mynet, nodeid=345, layernames="trade", direction="both")
th.get_degree(mynet, nodeid=345, layernames="trade", direction="both")
th.get_random_alter(mynet, nodeid=345, layernames="trade", direction="both")
th.density(mynet, "kinship")

In [ ]:
th.components(mynet, "kinship", attrname="kinship_component")
th.degree(mynet, "kinship", attrname="kinship_degree", direction="both")
th.get_attr_summary(mynet, "kinship_degree")

In [ ]:
th.shortest_path(mynet, node1id=123, node2id=567, layernames="kinship")
paths_by_gender = th.shortest_paths("paths_by_gender", mynet, attrname="gender", layernames="kinship")
th.info(paths_by_gender)

## Two-mode layers

The `work` layer is a two-mode affiliation layer. Hyperedge helpers use `hypername` for the affiliation name.

In [ ]:
th.get_all_hyperedges(mynet, "work", offset=0, limit=10)
th.get_hyperedge_nodes(mynet, "work", hypername="ias")
th.get_node_hyperedges(mynet, "work", nodeid=123)

In [ ]:
th.add_hyper(mynet, "work", hypername="py_temp", nodes=[123, 234])
th.add_aff(mynet, "work", nodeid=345, hypername="py_temp")
th.get_hyperedge_nodes(mynet, "work", hypername="py_temp")

th.remove_aff(mynet, "work", nodeid=345, hypername="py_temp")
th.remove_hyper(mynet, "work", hypername="py_temp")

## Derived structures and transformations

In [ ]:
female_nodeset = th.filter("female_nodeset", mynet_nodeset, attrname="gender", cond="eq", attrvalue="f")
female_mynet = th.subnet("female_mynet", mynet, female_nodeset)
th.info(female_nodeset), th.info(female_mynet)

In [ ]:
th.symmetrize(mynet, "trade", method="max", newlayername="trade_sym")
th.dichotomize(mynet, "trade", cond="ge", threshold=1000, truevalue=1, falsevalue=0, newlayername="trade_hi")
th.project_two_mode(mynet, "work", method="count", newlayername="work_projected")

th.get_edge(mynet, "trade_sym", 123, 345)
th.get_edge(mynet, "trade_hi", 890, 234)
th.get_edge(mynet, "work_projected", 123, 456)

## Scratch network editing

Use scratch structures when learning mutating commands. This keeps the bundled example network easy to inspect.

In [ ]:
scratch_nodes = th.create_nodeset("scratch_nodes", name="Scratch nodes", createnodes=5)
scratch_net = th.create_network("scratch_net", scratch_nodes, name="Scratch network")

th.add_layer(scratch_net, "friendship", mode=1, directed=False, valuetype="binary")
th.add_edge(scratch_net, "friendship", node1id=1, node2id=2)
th.check_edge(scratch_net, "friendship", node1id=1, node2id=2)
th.remove_edge(scratch_net, "friendship", node1id=1, node2id=2)
th.clear_layer(scratch_net, "friendship")

In [ ]:
th.add_node(scratch_nodes, nodeid=10)
th.get_all_nodes(scratch_nodes, offset=0, limit=20)
th.remove_node(scratch_nodes, nodeid=10)

th.define_attr(scratch_nodes, "score", "int")
th.set_attr(scratch_nodes, nodeid=1, attrname="score", attrvalue=7)
th.get_attr(scratch_nodes, nodeid=1, attrname="score")
th.remove_attr(scratch_nodes, nodeid=1, attrname="score")
th.undefine_attr(scratch_nodes, "score")

## Import and export

In [ ]:
exported_file = os.path.join(tutorial_workdir, "kinship_export.tsv")
import_file = os.path.join(tutorial_workdir, "imported_edges.tsv")
saved_nodes_file = os.path.join(tutorial_workdir, "scratch_nodes_saved.tsv")

th.export_layer(mynet, "kinship", file=exported_file, header=True, sep="\t")
with open(exported_file, encoding="utf-8") as f:
    print(f.read().splitlines()[:3])

with open(import_file, "w", encoding="utf-8") as f:
    f.write("1\t2\t4\n2\t3\t5\n")

th.add_layer(scratch_net, "imported_edges", mode=1, directed=False, valuetype="valued")
th.import_layer(scratch_net, "imported_edges", file=import_file, format="edgelist", header=False, addmissingnodes=True)
print(th.get_edge(scratch_net, "imported_edges", node1id=1, node2id=2))

th.save_file(scratch_nodes, file=saved_nodes_file)
scratch_copy = th.load_file("scratch_nodes_copy", file=saved_nodes_file, type="nodeset")
print(th.get_nbr_nodes(scratch_copy))

In [ ]:
# Remove files created by the import/export example.
for file in [exported_file, import_file, saved_nodes_file]:
    try:
        os.remove(file)
    except FileNotFoundError:
        pass

## Low-level commands and cleanup

In [ ]:
th.cmd("info", args={"structure": mynet})
th.delete_all()
th.i()

In [ ]:
threadlepy.stop()